# TCGA-BRCA WSI Download and TNBC Prediction

This notebook prepares the BRCA whole-slide image predictions used by the empirical example. It requires large local storage, network access to GDC, OpenSlide, and GPU time for model training, so it's recommended to run this on a computing cluster.

The workflow:

1. query GDC for TCGA-BRCA slide-image metadata;
2. link slide files to TCGA patient barcodes and TNBC labels;
3. download and validate available SVS files;
4. train a weakly supervised ResNet18 MIL classifier;
5. write patient-level predicted probabilities and uncertainty scores.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import io
import json
import os
import random
import shutil
import sys
import tarfile
import time
import warnings

import numpy as np
import pandas as pd
import requests
from PIL import Image
from sklearn.model_selection import train_test_split

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {"BRCA", "CheXpert", "Alphafold", "Stance", "Simulation"} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
    from torchvision.models import ResNet18_Weights, resnet18

    HAS_TORCH = True
    TORCH_IMPORT_ERROR = None
except Exception as exc:
    HAS_TORCH = False
    TORCH_IMPORT_ERROR = exc

try:
    import openslide

    HAS_OPENSLIDE = True
    OPENSLIDE_IMPORT_ERROR = None
except Exception as exc:
    HAS_OPENSLIDE = False
    OPENSLIDE_IMPORT_ERROR = exc


## Configuration

Set the environment variable `TCGA_BRCA_WSI_PROJECT_DIR` (raw WSI data is large, so it's recommended to use a cluster scratch path).


In [ ]:
EXAMPLE_DIR = REPO_ROOT / "BRCA"
DATA_DIR = REPO_ROOT / "Data" / "BRCA"

PROJECT_DIR = Path(
    os.environ.get("TCGA_BRCA_WSI_PROJECT_DIR", "~/tcga_brca_wsi_project")
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
DOWNLOAD_DIR = PROJECT_DIR / "downloads"
SVS_DIR = PROJECT_DIR / "svs"
PATCH_CACHE_DIR = PROJECT_DIR / "patch_cache"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [METADATA_DIR, DOWNLOAD_DIR, SVS_DIR, PATCH_CACHE_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PATIENT_COL = "case_submitter_id"
LABEL_COL = "is_tnbc"
CLINICAL_LABEL_CSV = DATA_DIR / "master_df.csv"

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
GDC_CASES_URL = "https://api.gdc.cancer.gov/cases"
GDC_TOKEN_PATH = None

SAMPLE_N_FILES = 20
DOWNLOAD_TIMEOUT = 120
MAX_DOWNLOAD_RETRIES = 5
SLEEP_BETWEEN_RETRIES = 5

NUM_PATCHES = 16
PATCH_SIZE = 224
WSI_LEVEL = 0
WHITE_THRESHOLD = 220

BATCH_SIZE = 4
NUM_EPOCHS = 5
LR = 1e-3
TRAIN_FRACTION = 0.80
MAX_TRAIN_PATIENTS = 50
SEED = 614
WRITE_REPO_PREDICTIONS = False

print("Project directory:", PROJECT_DIR)
print("Torch available:", HAS_TORCH)
print("OpenSlide available:", HAS_OPENSLIDE)
if HAS_TORCH:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", DEVICE)
else:
    DEVICE = None
    print("Torch import error:", repr(TORCH_IMPORT_ERROR))
if not HAS_OPENSLIDE:
    print("OpenSlide import error:", repr(OPENSLIDE_IMPORT_ERROR))


In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


set_seed(SEED)


## GDC Slide Manifest
 
Query GDC metadata (does not download SVS image files).


In [ ]:
def post_gdc_tsv(url: str, payload: dict, timeout: int = DOWNLOAD_TIMEOUT) -> pd.DataFrame:
    """POST to a GDC endpoint and return a TSV response as a DataFrame."""

    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=timeout,
    )
    if response.status_code != 200:
        raise RuntimeError(f"GDC API error {response.status_code}:\n{response.text[:1000]}")
    if not response.text.strip():
        raise ValueError("GDC query returned an empty response. Check filters.")
    return pd.read_csv(io.StringIO(response.text), sep="\t")


slide_payload = {
    "filters": {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]},
            },
            {
                "op": "in",
                "content": {"field": "files.data_category", "value": ["biospecimen"]},
            },
            {
                "op": "in",
                "content": {"field": "files.data_type", "value": ["Slide Image"]},
            },
        ],
    },
    "format": "TSV",
    "size": 20000,
    "fields": "file_id,file_name,cases.case_id",
}

manifest_df = post_gdc_tsv(GDC_FILES_URL, slide_payload)
manifest_path = METADATA_DIR / "tcga_brca_slide_manifest_raw.tsv"
manifest_df.to_csv(manifest_path, sep="\t", index=False)

print("Slide-image rows:", manifest_df.shape)
display(manifest_df.head())
print("Saved:", manifest_path)


In [ ]:
case_payload = {
    "filters": {
        "op": "in",
        "content": {"field": "project.project_id", "value": ["TCGA-BRCA"]},
    },
    "format": "TSV",
    "size": 2500,
    "fields": "case_id,submitter_id",
}

case_map = post_gdc_tsv(GDC_CASES_URL, case_payload)
case_map = case_map.rename(columns={"submitter_id": PATIENT_COL})

case_map_path = METADATA_DIR / "tcga_brca_case_map.tsv"
case_map.to_csv(case_map_path, sep="\t", index=False)

print("Case map:", case_map.shape)
display(case_map.head())
print("Saved:", case_map_path)


In [ ]:
def link_manifest_to_patients(manifest: pd.DataFrame, cases: pd.DataFrame) -> pd.DataFrame:
    """Attach TCGA patient barcodes to a GDC slide manifest."""

    case_id_columns = ["cases.0.case_id", "cases.case_id", "case_id"]
    case_id_col = next((col for col in case_id_columns if col in manifest.columns), None)
    if case_id_col is None:
        raise ValueError(f"No case identifier column found in manifest: {manifest.columns.tolist()}")

    slide_df = manifest.rename(columns={case_id_col: "case_id"}).copy()
    linked = slide_df.merge(cases, on="case_id", how="left")
    linked = linked.dropna(subset=[PATIENT_COL]).copy()
    linked = linked.sort_values([PATIENT_COL, "file_name", "file_id"]).reset_index(drop=True)
    return linked.drop_duplicates(subset=PATIENT_COL, keep="first").copy()


slide_unique_df = link_manifest_to_patients(manifest_df, case_map)
linked_path = METADATA_DIR / "tcga_brca_slide_manifest_patient_linked.csv"
slide_unique_df.to_csv(linked_path, index=False)

print("Unique patients with slides:", slide_unique_df.shape)
display(slide_unique_df.head())
print("Saved:", linked_path)


## Clinical Labels

The default clinical table is `Data/BRCA/master_df.csv`, which already contains the TNBC label used in the example.  Any existing file metadata columns are ignored before re-linking to the current GDC manifest.


In [ ]:
def load_clinical_labels(path: Path = CLINICAL_LABEL_CSV) -> pd.DataFrame:
    clinical = pd.read_csv(path)
    required = {PATIENT_COL, LABEL_COL}
    missing = required - set(clinical.columns)
    if missing:
        raise ValueError(f"Clinical table is missing required columns: {missing}")

    metadata_cols = {"file_id", "file_name", "case_id", "svs_path", "svs_available"}
    clinical = clinical.drop(columns=[col for col in metadata_cols if col in clinical.columns])
    clinical[LABEL_COL] = clinical[LABEL_COL].astype(int)
    return clinical.drop_duplicates(subset=PATIENT_COL).copy()


clinical_df = load_clinical_labels()
master_df = clinical_df.merge(slide_unique_df, on=PATIENT_COL, how="inner")
master_df[LABEL_COL] = master_df[LABEL_COL].astype(int)

master_path = METADATA_DIR / "tcga_brca_master_table.csv"
master_df.to_csv(master_path, index=False)

print("Final master table:", master_df.shape)
print(master_df[LABEL_COL].value_counts(dropna=False))
display(master_df.head())
print("Saved:", master_path)


## Download SVS Files

Download helpers. Completed files are skipped, so it's safe to rerun `download_gdc_file()` can be rerun.


In [ ]:
def get_gdc_headers() -> dict:
    headers = {}
    if GDC_TOKEN_PATH is not None and Path(GDC_TOKEN_PATH).exists():
        headers["X-Auth-Token"] = Path(GDC_TOKEN_PATH).read_text().strip()
    return headers


def expected_download_path(file_id: str) -> Path:
    return DOWNLOAD_DIR / str(file_id)


def file_already_downloaded(file_id: str) -> bool:
    folder = expected_download_path(file_id)
    if (SVS_DIR / f"{file_id}.svs").exists():
        return True
    return folder.exists() and any(folder.rglob("*"))


def download_gdc_file(
    file_id: str,
    file_name: str | None = None,
    overwrite: bool = False,
    chunk_size: int = 1024 * 1024,
) -> dict:
    """Download one GDC file into DOWNLOAD_DIR / file_id."""

    out_dir = expected_download_path(file_id)
    out_dir.mkdir(parents=True, exist_ok=True)

    if file_already_downloaded(file_id) and not overwrite:
        return {"file_id": file_id, "status": "skipped_exists", "path": str(out_dir)}

    url = f"https://api.gdc.cancer.gov/data/{file_id}"
    last_error = None
    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
        try:
            with requests.get(url, headers=get_gdc_headers(), stream=True, timeout=DOWNLOAD_TIMEOUT) as response:
                if response.status_code != 200:
                    raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

                content_disposition = response.headers.get("Content-Disposition", "")
                inferred_name = None
                if "filename=" in content_disposition:
                    inferred_name = content_disposition.split("filename=")[-1].strip().strip('"')
                if inferred_name is None:
                    inferred_name = file_name if file_name else file_id

                out_path = out_dir / inferred_name
                tmp_path = out_dir / f"{inferred_name}.part"
                with open(tmp_path, "wb") as handle:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            handle.write(chunk)
                tmp_path.replace(out_path)
                return {"file_id": file_id, "status": "downloaded", "path": str(out_path)}
        except Exception as exc:
            last_error = repr(exc)
            print(f"[{file_id}] attempt {attempt}/{MAX_DOWNLOAD_RETRIES} failed: {last_error}")
            time.sleep(SLEEP_BETWEEN_RETRIES)

    return {"file_id": file_id, "status": "failed", "error": last_error}


def make_download_table(master: pd.DataFrame, sample_n_files: int | None = SAMPLE_N_FILES) -> pd.DataFrame:
    table = master.copy()
    if sample_n_files is not None:
        table = table.head(sample_n_files).copy()
    return table


download_table = make_download_table(master_df)
print("Files scheduled for download:", len(download_table))
display(download_table[[PATIENT_COL, "file_id", "file_name"]].head())


In [ ]:
download_results = []
for _, row in download_table.iterrows():
    result = download_gdc_file(
        file_id=row["file_id"],
        file_name=row.get("file_name"),
        overwrite=False,
    )
    download_results.append(result)
    print(result["status"], result["file_id"])

download_results_df = pd.DataFrame(download_results)
download_results_path = METADATA_DIR / "download_results_latest.csv"
download_results_df.to_csv(download_results_path, index=False)

print(download_results_df["status"].value_counts(dropna=False))
display(download_results_df.head())
print("Saved:", download_results_path)


## Extract and Validate SVS Files


In [ ]:
def extract_svs_for_file_id(file_id: str) -> dict:
    """Find or extract one SVS file for a GDC file_id."""

    out_svs = SVS_DIR / f"{file_id}.svs"
    if out_svs.exists() and out_svs.stat().st_size > 0:
        return {"file_id": file_id, "status": "svs_exists", "svs_path": str(out_svs)}

    folder = expected_download_path(file_id)
    if not folder.exists():
        return {"file_id": file_id, "status": "download_folder_missing"}

    svs_files = list(folder.rglob("*.svs"))
    if svs_files:
        shutil.copy2(svs_files[0], out_svs)
        return {"file_id": file_id, "status": "copied_svs", "svs_path": str(out_svs)}

    archives = []
    for pattern in ["*.tar.gz", "*.tgz", "*.tar"]:
        archives.extend(folder.rglob(pattern))

    for archive in archives:
        try:
            mode = "r:gz" if archive.name.endswith((".tar.gz", ".tgz")) else "r:"
            with tarfile.open(archive, mode) as tar:
                members = [member for member in tar.getmembers() if member.name.lower().endswith(".svs")]
                if not members:
                    continue
                extracted = tar.extractfile(members[0])
                if extracted is None:
                    continue
                with open(out_svs, "wb") as out:
                    shutil.copyfileobj(extracted, out)
                return {"file_id": file_id, "status": "extracted_svs", "svs_path": str(out_svs)}
        except Exception as exc:
            warnings.warn(f"Failed to extract {archive}: {repr(exc)}")

    return {"file_id": file_id, "status": "no_svs_found"}


extract_results = []
for file_id in download_table["file_id"].tolist():
    result = extract_svs_for_file_id(file_id)
    extract_results.append(result)
    print(result["status"], file_id)

extract_results_df = pd.DataFrame(extract_results)
extract_results_path = METADATA_DIR / "extract_results_latest.csv"
extract_results_df.to_csv(extract_results_path, index=False)

print(extract_results_df["status"].value_counts(dropna=False))
display(extract_results_df.head())
print("Saved:", extract_results_path)


In [ ]:
def validate_available_svs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["svs_path"] = [str(SVS_DIR / f"{file_id}.svs") for file_id in out["file_id"]]
    out["svs_available"] = [Path(path).exists() and Path(path).stat().st_size > 0 for path in out["svs_path"]]
    return out


master_with_svs = validate_available_svs(master_df)
available_df = master_with_svs[master_with_svs["svs_available"]].copy()
missing_df = master_with_svs[~master_with_svs["svs_available"]].copy()

available_path = METADATA_DIR / "master_with_available_svs.csv"
missing_path = METADATA_DIR / "missing_or_unretrieved_svs.csv"

available_df.to_csv(available_path, index=False)
missing_df[[PATIENT_COL, "file_id", "file_name"]].to_csv(missing_path, index=False)

print("Available SVS:", available_df.shape)
print("Missing/unretrieved SVS:", missing_df.shape)
display(missing_df[[PATIENT_COL, "file_id", "file_name"]].head())
print("Saved available:", available_path)
print("Saved missing:", missing_path)


## Patch Extraction and MIL Model

The model uses a frozen ImageNet ResNet18 feature extractor, averages patch-level features within a slide, and trains a patient-level binary classifier. These cells require OpenSlide and PyTorch.


In [ ]:
def require_torch_and_openslide() -> None:
    if not HAS_TORCH:
        raise ImportError(f"PyTorch/torchvision are required for model training: {TORCH_IMPORT_ERROR!r}")
    if not HAS_OPENSLIDE:
        raise ImportError(f"openslide-python is required for patch extraction: {OPENSLIDE_IMPORT_ERROR!r}")


def tissue_mask_from_thumbnail(slide, thumb_size: tuple[int, int] = (1024, 1024)) -> tuple[Image.Image, np.ndarray]:
    thumb = slide.get_thumbnail(thumb_size).convert("RGB")
    arr = np.asarray(thumb)
    return thumb, np.any(arr < WHITE_THRESHOLD, axis=-1)


def sample_patch_locations(
    slide,
    num_patches: int = NUM_PATCHES,
    patch_size: int = PATCH_SIZE,
) -> list[tuple[int, int]]:
    thumb, mask = tissue_mask_from_thumbnail(slide)
    ys, xs = np.where(mask)
    width0, height0 = slide.dimensions

    if len(xs) == 0:
        return [
            (
                random.randint(0, max(0, width0 - patch_size)),
                random.randint(0, max(0, height0 - patch_size)),
            )
            for _ in range(num_patches)
        ]

    thumb_w, thumb_h = thumb.size
    scale_x = width0 / thumb_w
    scale_y = height0 / thumb_h
    coords = []

    for _ in range(num_patches * 100):
        if len(coords) >= num_patches:
            break
        idx = random.randrange(len(xs))
        x0 = int(xs[idx] * scale_x)
        y0 = int(ys[idx] * scale_y)
        x0 = max(0, min(width0 - patch_size, x0 - patch_size // 2))
        y0 = max(0, min(height0 - patch_size, y0 - patch_size // 2))
        coords.append((x0, y0))

    while len(coords) < num_patches:
        coords.append(coords[-1] if coords else (0, 0))
    return coords


def extract_patches_from_wsi(
    svs_path: str | Path,
    num_patches: int = NUM_PATCHES,
    patch_size: int = PATCH_SIZE,
    level: int = WSI_LEVEL,
) -> list[Image.Image]:
    require_torch_and_openslide()
    slide = openslide.OpenSlide(str(svs_path))
    try:
        coords = sample_patch_locations(slide, num_patches=num_patches, patch_size=patch_size)
        return [
            slide.read_region((x0, y0), level, (patch_size, patch_size)).convert("RGB")
            for x0, y0 in coords
        ]
    finally:
        slide.close()


In [ ]:
require_torch_and_openslide()

resnet_weights = ResNet18_Weights.DEFAULT
patch_transform = resnet_weights.transforms()


class WsiTnbcDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        patient_col: str = PATIENT_COL,
        label_col: str = LABEL_COL,
        svs_col: str = "svs_path",
        num_patches: int = NUM_PATCHES,
        patch_size: int = PATCH_SIZE,
        transform=patch_transform,
        cache_dir: Path | None = None,
    ) -> None:
        self.df = df.reset_index(drop=True).copy()
        self.patient_col = patient_col
        self.label_col = label_col
        self.svs_col = svs_col
        self.num_patches = num_patches
        self.patch_size = patch_size
        self.transform = transform
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None
        if self.cache_dir is not None:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

    def __len__(self) -> int:
        return len(self.df)

    def _cache_path(self, patient_id: str) -> Path:
        safe_id = str(patient_id).replace("/", "_")
        return self.cache_dir / f"{safe_id}_patches.pt"

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        patient_id = row[self.patient_col]
        label = torch.tensor(float(row[self.label_col]), dtype=torch.float32)

        if self.cache_dir is not None:
            cache_path = self._cache_path(patient_id)
            if cache_path.exists():
                return torch.load(cache_path, map_location="cpu"), label, patient_id

        patches = extract_patches_from_wsi(
            svs_path=row[self.svs_col],
            num_patches=self.num_patches,
            patch_size=self.patch_size,
        )
        patch_tensor = torch.stack([self.transform(patch) for patch in patches], dim=0)

        if self.cache_dir is not None:
            torch.save(patch_tensor, cache_path)
        return patch_tensor, label, patient_id


class WsiMilModel(nn.Module):
    def __init__(self, freeze_backbone: bool = True) -> None:
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.feature_extractor = backbone
        self.classifier = nn.Linear(in_features, 1)

        if freeze_backbone:
            for parameter in self.feature_extractor.parameters():
                parameter.requires_grad = False

    def forward(self, x):
        batch_size, num_patches, channels, height, width = x.shape
        x_flat = x.view(batch_size * num_patches, channels, height, width)
        features = self.feature_extractor(x_flat).view(batch_size, num_patches, -1)
        pooled = features.mean(dim=1)
        return self.classifier(pooled).squeeze(-1)


## Train/Test Split


In [ ]:
def subset_for_training(df: pd.DataFrame, max_patients: int | None = MAX_TRAIN_PATIENTS) -> pd.DataFrame:
    model_df = df[df[LABEL_COL].notna()].copy()
    model_df[LABEL_COL] = model_df[LABEL_COL].astype(int)
    if max_patients is None or len(model_df) <= max_patients:
        return model_df.reset_index(drop=True)

    parts = []
    n_classes = max(1, model_df[LABEL_COL].nunique())
    for _, group in model_df.groupby(LABEL_COL):
        n_take = max(1, min(len(group), max_patients // n_classes))
        parts.append(group.sample(n=n_take, random_state=SEED))
    sampled = pd.concat(parts, ignore_index=True)
    if len(sampled) > max_patients:
        sampled = sampled.sample(n=max_patients, random_state=SEED)
    return sampled.reset_index(drop=True)


model_df = subset_for_training(available_df)
stratify = model_df[LABEL_COL] if model_df[LABEL_COL].nunique() > 1 and model_df[LABEL_COL].value_counts().min() >= 2 else None
train_df, val_df = train_test_split(
    model_df,
    train_size=TRAIN_FRACTION,
    random_state=SEED,
    stratify=stratify,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_ds = WsiTnbcDataset(train_df, cache_dir=PATCH_CACHE_DIR / "train")
val_ds = WsiTnbcDataset(val_df, cache_dir=PATCH_CACHE_DIR / "val")

class_counts = train_df[LABEL_COL].value_counts().to_dict()
sample_weights = train_df[LABEL_COL].map(lambda label: 1.0 / class_counts[int(label)]).to_numpy()
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Modeling cohort:", model_df.shape)
print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Val:", val_df.shape, val_df[LABEL_COL].value_counts().to_dict())


## Train WSI Classifier


In [ ]:
model = WsiMilModel(freeze_backbone=True).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LR)


def run_epoch(loader: DataLoader, train: bool = True) -> tuple[float, float, pd.DataFrame]:
    model.train(mode=train)
    losses = []
    all_probs = []
    all_labels = []
    all_patients = []

    for patches, labels, patient_ids in loader:
        patches = patches.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(patches)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        losses.append(loss.item())
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_patients.extend(list(patient_ids))

    out = pd.DataFrame(
        {
            PATIENT_COL: all_patients,
            "true_label_from_loader": all_labels,
            "predicted_probability": all_probs,
        }
    )
    out["predicted_label"] = (out["predicted_probability"] >= 0.5).astype(int)
    out["is_correct"] = out["predicted_label"] == out["true_label_from_loader"]
    accuracy = out["is_correct"].mean() if len(out) else np.nan
    return float(np.mean(losses)) if losses else np.nan, accuracy, out


history = []
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, _ = run_epoch(train_loader, train=True)
    val_loss, val_acc, _ = run_epoch(val_loader, train=False)
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }
    history.append(row)
    print(row)

history_df = pd.DataFrame(history)
history_path = OUTPUT_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


## Predictions


In [ ]:
full_ds = WsiTnbcDataset(model_df.reset_index(drop=True), cache_dir=PATCH_CACHE_DIR / "full")
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

_, _, pred_df = run_epoch(full_loader, train=False)
pred_df["uncertainty"] = pred_df["predicted_probability"] * (1.0 - pred_df["predicted_probability"])

pred_out = pred_df.merge(
    model_df[[PATIENT_COL, "file_id", "file_name", "case_id", "svs_path", LABEL_COL]],
    on=PATIENT_COL,
    how="left",
)
if "race" in model_df.columns:
    pred_out = pred_out.merge(model_df[[PATIENT_COL, "race"]], on=PATIENT_COL, how="left")
    pred_out["strat"] = pred_out["race"] + "_" + pred_out[LABEL_COL].map({1: "TNBC", 0: "nonTNBC"})

pred_path = OUTPUT_DIR / "tcga_brca_wsi_predictions.csv"
pred_out.to_csv(pred_path, index=False)

if WRITE_REPO_PREDICTIONS:
    repo_pred_path = DATA_DIR / "master_df_withpred.csv"
    pred_out.to_csv(repo_pred_path, index=False)
    print("Saved repo prediction table:", repo_pred_path)

display(pred_out.head())
print("Saved:", pred_path)


In [ ]:
checkpoint_path = OUTPUT_DIR / "wsi_mil_resnet18_tnbc.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "num_patches": NUM_PATCHES,
            "patch_size": PATCH_SIZE,
            "wsi_level": WSI_LEVEL,
            "label_col": LABEL_COL,
            "patient_col": PATIENT_COL,
            "freeze_backbone": True,
            "seed": SEED,
        },
        "history": history,
    },
    checkpoint_path,
)
print("Saved:", checkpoint_path)


## Missing-File Report


In [ ]:
missing_report_cols = [PATIENT_COL, "file_id", "file_name"]
missing_report = missing_df[missing_report_cols].copy()
missing_report["reason_for_exclusion"] = "unavailable or incomplete after repeated download attempts"

missing_report_path = OUTPUT_DIR / "gdc_download_failures_for_appendix.csv"
missing_report.to_csv(missing_report_path, index=False)

print("Missing/unretrieved files:", missing_report.shape)
display(missing_report.head())
print("Saved:", missing_report_path)


## Scaling Notes

For a full run, set `SAMPLE_N_FILES = None` and `MAX_TRAIN_PATIENTS = None`, then run the download/extraction cells until the missing-file set stops shrinking. Training should be run on a GPU-enabled machine.

The experiment notebook (`BRCA_active_clean.ipynb`) expects `Data/BRCA/master_df_withpred.csv`. Keep `WRITE_REPO_PREDICTIONS = False` while experimenting, then explicitly save the final prediction table when you are ready to replace that input.
